# Registering optodes using the spring-relaxation method

In [ ]:
import cedalion
import cedalion.data
import cedalion.dot
import xarray as xr
import numpy as np
import pyvista as pv
import cedalion.vis.blocks as vbx
from cedalion.vis.anatomy import scalp_plot
import matplotlib.pyplot as p
from cedalion.nirs import channel_distances
from cedalion.geometry.landmarks import normalize_landmarks_labels

pv.set_jupyter_backend("static")

## Preparing datasets

In [ ]:
DATASETS = [
    "fingertapping",
    "fingertappingDOT",
    "nn22_resting",
    "ninja_cap_56x144",
    "ninja_uhd_cap_164x496",
    "lumo",
    "kernel",
    "artinis",
]

HEADMODELS = ["icbm152"]  # + ["colin27"]


def get_fnirs_dataset(dataset):
    if dataset == "fingertappingDOT":
        rec = cedalion.data.get_fingertappingDOT()
        return rec.geo3d, rec._measurement_lists["amp"], rec["amp"]

    elif dataset == "fingertapping":
        rec = cedalion.data.get_fingertapping()
        return rec.geo3d, rec._measurement_lists["amp"], rec["amp"]

    elif dataset == "nn22_resting":
        rec = cedalion.data.get_nn22_resting_state()
        return rec.geo3d, rec._measurement_lists["amp"], rec["amp"]

    elif dataset == "ninja_cap_56x144":
        geo3d, landmarks, meas_list = cedalion.data.get_ninja_cap_probe()
        geo3d = xr.concat((geo3d, landmarks), dim="label")
        geo3d = geo3d.pint.quantify("mm")
        ts = cedalion.dataclasses.empty_timeseries_from_measurement_list(meas_list)
        return geo3d, meas_list, ts

    elif dataset == "ninja_uhd_cap_164x496":
        geo3d, landmarks, meas_list = cedalion.data.get_ninja_uhd_cap_probe()
        geo3d = xr.concat((geo3d, landmarks), dim="label")
        geo3d = geo3d.pint.quantify("mm")
        ts = cedalion.dataclasses.empty_timeseries_from_measurement_list(meas_list)
        return geo3d, meas_list, ts

    elif dataset == "lumo":
        rec = cedalion.data.get_lumo_testdataset()
        geo3d = normalize_landmarks_labels(rec.geo3d)

        ch_mask = (
            cedalion.nirs.channel_distances(rec["amp"], geo3d) < 3.5 * cedalion.units.cm
        )
        rec["amp"] = rec["amp"].sel(channel=ch_mask)

        return geo3d, rec._measurement_lists["amp"], rec["amp"]

    elif dataset == "artinis":
        rec = cedalion.data.get_artinis_testdataset()
        geo3d = normalize_landmarks_labels(rec.geo3d)
        geo3d = geo3d.pint.dequantify().pint.quantify("cm").pint.to("mm")

        # dataset lacks landmarks but is in MNI305. Adopt Colin landmarks
        head_ijk = cedalion.dot.get_standard_headmodel("colin27")
        head_ras = head_ijk.apply_transform(head_ijk.t_ijk2ras)
        colin_landmarks = head_ras.landmarks.sel(label=["Nz", "Iz", "LPA", "RPA", "Cz"])
        colin_landmarks = colin_landmarks.points.set_crs("pos")

        geo3d = xr.concat(
            (geo3d, colin_landmarks),
            dim="label",
        )

        return geo3d, rec._measurement_lists["amp"], rec["amp"]

    elif dataset == "kernel":
        rec = cedalion.data.get_kernel_testdataset()
        geo3d = normalize_landmarks_labels(rec.geo3d)
        return geo3d, rec._measurement_lists["amp"], rec["amp"]

## Registering and Visualizing Results

For each dataset and headmodel register via `align_and_snap_to_scalp` and `align_and_relax_to_scalp`.

Plot the registered montages and show how much channel distances changed through the registration.

In [ ]:
for dataset in DATASETS:
    for headmodel in HEADMODELS:
        print(dataset, headmodel)
        head_ijk = cedalion.dot.get_standard_headmodel(headmodel)
        head_ras = head_ijk.apply_transform(head_ijk.t_ijk2ras)

        geo3d, meas_list, ts = get_fnirs_dataset(dataset)

        common_landmarks = geo3d.points.common_labels(head_ijk.landmarks)

        if len(common_landmarks) > 3:
            initial_align_mode = "general"
        elif len(common_landmarks) == 3:
            initial_align_mode = "trans_rot_isoscale"
        else:
            initial_align_mode = "identity"

        # align and snap to scalp
        geo3d_snapped = head_ras.align_and_snap_to_scalp(geo3d, mode=initial_align_mode)

        # align and relax to scalp - needs channel information. here provided through NDTimeSeries
        geo3d_relaxed, details = head_ras.align_and_relax_to_scalp(
            geo3d, ts, initial_align_mode=initial_align_mode
        )

        # visualize results
        plt = pv.Plotter(shape=(1, 2), window_size=(900,450))

        plt.subplot(0,0)
        vbx.plot_surface(plt, head_ras.scalp, color="w")
        vbx.plot_labeled_points(plt, geo3d_snapped)
        plt.add_text("align_and_snap_to_scalp", font_size=8)
        plt.subplot(0, 1)
        vbx.plot_surface(plt, head_ras.scalp, color="w")
        vbx.plot_labeled_points(plt, geo3d_relaxed)
        plt.add_text("align_and_relax_to_scalp", font_size=8)
        plt.show()

        f,ax = p.subplots(1,2, figsize=(9,4.5), dpi=100)
        nominal_distances = channel_distances(ts, geo3d)
        d_snapped = (channel_distances(ts, geo3d_snapped) - nominal_distances).pint.to("mm")
        d_relaxed = (channel_distances(ts, geo3d_relaxed) - nominal_distances).pint.to("mm")
        spargs = dict(
            vmin=-10,
            vmax=10,
            optode_size=2,
            cb_label="$\Delta$ ch. distance / mm",
            cmap=p.cm.RdYlBu_r
        )
        scalp_plot(ts, geo3d_snapped, d_snapped, ax[0], **spargs)
        scalp_plot(ts, geo3d_relaxed, d_relaxed, ax[1], **spargs)
        f.suptitle(f"{dataset} - {headmodel}")
        p.tight_layout()
        display(f)
        p.close(f)